# CH14 — Grafos (parte 2): recorridos BFS y DFS, componentes conexas y laberintos

Material del curso basado en Goodrich, Tamassia & Goldwasser — Capítulo 14 (secciones 14.3.1 a 14.3.4).

**Objetivo:** entender el recorrido en **anchura** (BFS) como una exploración **por niveles**, usarlo para encontrar las **componentes conexas** de un grafo y para resolver un **laberinto** por el camino más corto. Todo se ejecuta con el código original de Goodrich (`goodrich/ch14`).

**Contenido**

0. Preparación: importar el código de Goodrich
1. Componentes conexas
2. BFS: recorrido por niveles
3. `BFS_complete`: encontrar todas las componentes
4. DFS: el otro recorrido
5. Aplicación: salir de un laberinto
6. Resumen y ejercicios

## 0. Preparación: importar el código de Goodrich

En el notebook de teoría escribimos la clase `Graph` a mano. Aquí **no** la copiamos: la importamos del paquete `goodrich` que está en la raíz del repositorio. La celda siguiente busca esa carpeta subiendo desde el directorio del notebook y la agrega a `sys.path`, así funciona en cualquier computador sin escribir rutas fijas.

| Archivo | Qué trae |
|---|---|
| `goodrich/ch14/graph.py` | la clase `Graph` (mapa de adyacencia, Códigos 14.1–14.3) |
| `goodrich/ch14/bfs.py` | `BFS(g, s, discovered)` y `BFS_complete(g)` (Código 14.9) |
| `goodrich/ch14/dfs.py` | `DFS(g, u, discovered)`, `construct_path(u, v, discovered)` y `DFS_complete(g)` (Códigos 14.5–14.8) |

In [13]:
import sys
sys.path.append(r'C:\Users\fator\Desktop\Proyectos1\Estructura_Datos\2026_02')

In [10]:
from goodrich.ch14.graph import Graph
from goodrich.ch14.bfs import BFS, BFS_complete
from goodrich.ch14.dfs import DFS, DFS_complete, construct_path

## 1. Componentes conexas

Recordemos (sección 1 del notebook de teoría): un grafo no dirigido es **conexo** si hay un camino entre **todo** par de vértices.

Cuando un grafo **no** es conexo, se parte en pedazos: las **componentes conexas**. Una componente conexa es un conjunto **maximal** de vértices que están conectados entre sí:

- dentro de una componente, cualquier vértice alcanza a cualquier otro;
- entre componentes distintas **no hay ninguna arista** (si la hubiera, serían la misma componente);
- cada vértice está en **exactamente una** componente. Un vértice aislado (grado 0) es una componente él solo.

El grafo de la figura tiene 13 vértices y 12 aristas, y se parte en **4 componentes**. Abajo están coloreadas tal como las encuentra `BFS_complete` de Goodrich (sección 3).

![Grafo con cuatro componentes conexas](../assets/ch14_componentes.png)

**¿Dónde aparecen?** Grupos de amigos que no se conocen entre sí en una red social, zonas de un laberinto separadas por muros, islas de una red eléctrica después de un apagón, "clusters" de páginas web...

Construimos el grafo con la clase `Graph` de Goodrich. La función `graph_from_edgelist` de `graph_examples.py` **no** sirve aquí por dos razones: no permite vértices aislados (como `M`), y guarda las etiquetas en un `set`, cuyo orden cambia entre sesiones de Python. Por eso escribimos una versión pequeña que conserva el orden de inserción y retorna el diccionario `etiqueta -> Vertex`.

In [14]:
def construir(aristas, aislados=()):
    """Construye un Graph (no dirigido) de Goodrich.

    aristas: tuplas (u, v) de etiquetas. aislados: etiquetas de vértices sin aristas.
    Retorna (g, V) donde V es un dict etiqueta -> Vertex.
    """
    g = Graph()
    V = {}
    for u, v in aristas:
        for x in (u, v):
            if x not in V:
                V[x] = g.insert_vertex(x)
    for x in aislados:
        V[x] = g.insert_vertex(x)
    for u, v in aristas:
        g.insert_edge(V[u], V[v])
    return g, V


ARISTAS = [('A', 'B'), ('A', 'C'), ('B', 'D'), ('C', 'D'), ('C', 'E'), ('D', 'F'), ('E', 'F'),   # componente 1
           ('G', 'H'), ('G', 'I'), ('H', 'I'), ('I', 'J'),                                        # componente 2
           ('K', 'L')]                                                                           # componente 3
g, V = construir(ARISTAS, aislados=['M'])                                                        # componente 4

print(g.vertex_count(), 'vértices,', g.edge_count(), 'aristas')
print('grado de M:', g.degree(V['M']))

13 vértices, 12 aristas
grado de M: 0


## 2. BFS: recorrido por niveles

El recorrido en anchura (*breadth-first search*) parte de un vértice $s$ y avanza como una **onda** que se expande:

- **nivel 0:** solo $s$;
- **nivel 1:** los vecinos de $s$;
- **nivel 2:** los vecinos de los del nivel 1 que **aún no** se han visitado;
- ... y así hasta que un nivel quede vacío.

El nivel de un vértice es su **distancia** a $s$ medida en número de aristas: BFS visita primero todo lo que está a distancia 1, luego todo lo que está a distancia 2, etc.

![BFS por niveles desde A](../assets/ch14_bfs_niveles.png)

Observa dos cosas en la figura:

1. Las aristas verdes forman un **árbol BFS**: cada vértice cuelga de la arista por la que fue **descubierto**. Las aristas punteadas (`C–D`, `E–F`) llevan a un vértice que **ya** estaba descubierto y por eso no entran al árbol.
2. BFS desde `A` **solo** visita la componente de `A`. No hay ningún camino hacia `G, ..., M`, así que la onda nunca llega allá.

### El código de Goodrich (Código 14.9)

`BFS(g, s, discovered)` no retorna nada: **llena** el diccionario `discovered`, que asocia cada vértice descubierto con la **arista** que lo descubrió. Antes de llamarla hay que poner `discovered[s] = None` (la raíz no fue descubierta por ninguna arista).

En vez de una cola, Goodrich guarda el nivel actual en una lista `level` y arma la lista `next_level` con los vértices nuevos. Es equivalente a la versión con cola: los vértices se procesan en el mismo orden.

In [16]:
import inspect
print(inspect.getsource(BFS))

def BFS(g, s, discovered):
  """Perform BFS of the undiscovered portion of Graph g starting at Vertex s.

  discovered is a dictionary mapping each vertex to the edge that was used to
  discover it during the BFS (s should be mapped to None prior to the call).
  Newly discovered vertices will be added to the dictionary as a result.
  """
  level = [s]                        # first level includes only s
  while len(level) > 0:
    next_level = []                  # prepare to gather newly found vertices
    for u in level:
      for e in g.incident_edges(u):  # for every outgoing edge from u
        v = e.opposite(u)
        if v not in discovered:      # v is an unvisited vertex
          discovered[v] = e          # e is the tree edge that discovered v
          next_level.append(v)       # v will be further considered in next pass
    level = next_level               # relabel 'next' level to become current



In [36]:
print(inspect.getsource(construct_path))

def construct_path(u, v, discovered):
  """
  Return a list of vertices comprising the directed path from u to v,
  or an empty list if v is not reachable from u.

  discovered is a dictionary resulting from a previous call to DFS started at u.
  """
  path = []                        # empty path by default
  if v in discovered:
    # we build list from v to u and then reverse it at the end
    path.append(v)
    walk = v
    while walk is not u:
      e = discovered[walk]         # find edge leading to walk
      parent = e.opposite(walk)
      path.append(parent)
      walk = parent
    path.reverse()                 # reorient path from u to v
  return path



In [33]:
def BFS1(g:Graph, v:Graph.Vertex, discovered:dict):
    procesar = [v]
    while procesar:
        current = procesar.pop(0)
        for i in g.incident_edges(current):
            opuesto = i.opposite(current)
            if opuesto not in discovered:
                discovered[opuesto] = current
                procesar.append(opuesto)

In [35]:
discovered = {V['A']:None}
BFS1(g, V['A'], discovered)
discovered

{A: None, B: A, C: A, D: B, E: C, F: D}

Ejecutamos `BFS` desde `A`. El diccionario resultante tiene **solo** los 6 vértices de la componente de `A`:

In [17]:
discovered = {V['A']: None}           # A es la raíz
BFS(g, V['A'], discovered)

for v, e in discovered.items():
    print(f'{v}  descubierto por  {e}')

print()
print('alcanzables desde A:', list(discovered))
print('¿alcanza a G?', V['G'] in discovered)

A  descubierto por  None
B  descubierto por  (A,B,None)
C  descubierto por  (A,C,None)
D  descubierto por  (B,D,None)
E  descubierto por  (C,E,None)
F  descubierto por  (D,F,None)

alcanzables desde A: [A, B, C, D, E, F]
¿alcanza a G? False


### Ver los niveles

`BFS` de Goodrich no guarda los niveles, pero los recorre uno por uno. La siguiente función es **el mismo algoritmo**, línea por línea, con una sola diferencia: además de llenar `discovered`, va guardando cada `level` en una lista para poder imprimirlos.

In [5]:
def BFS_niveles(g, s):
    """Igual que BFS de Goodrich, pero retorna (discovered, niveles)."""
    discovered = {s: None}
    niveles = []
    level = [s]
    while len(level) > 0:
        niveles.append(level)              # <- lo único nuevo
        next_level = []
        for u in level:
            for e in g.incident_edges(u):
                v = e.opposite(u)
                if v not in discovered:
                    discovered[v] = e
                    next_level.append(v)
        level = next_level
    return discovered, niveles


_, niveles = BFS_niveles(g, V['A'])
for k, nivel in enumerate(niveles):
    print(f'nivel {k}: {nivel}')

nivel 0: [A]
nivel 1: [B, C]
nivel 2: [D, E]
nivel 3: [F]


### Camino más corto con `construct_path`

Como cada vértice recuerda la arista que lo descubrió, se puede **devolver** desde cualquier vértice $v$ hasta la raíz siguiendo esas aristas. Eso hace `construct_path(u, v, discovered)` de `dfs.py`: aunque Goodrich la escribió para DFS, **sirve igual** con el diccionario de BFS, porque ambos guardan lo mismo (vértice -> arista de descubrimiento).

**Propiedad clave de BFS (Proposición 14.16):** el camino del árbol BFS de $s$ a $v$ es un camino con el **menor número de aristas** posible. La razón: $v$ se descubre en el primer nivel en el que aparece, y ese nivel es justo su distancia a $s$. Si el grafo es **no conexo**, `construct_path` retorna `[]` para los vértices de otras componentes.

In [6]:
discovered = {V['A']: None}
BFS(g, V['A'], discovered)

for destino in ['D', 'F', 'J']:
    camino = construct_path(V['A'], V[destino], discovered)
    if camino:
        print(f'A -> {destino}: {camino}  ({len(camino) - 1} aristas)')
    else:
        print(f'A -> {destino}: no hay camino (otra componente)')

A -> D: [A, B, D]  (2 aristas)
A -> F: [A, B, D, F]  (3 aristas)
A -> J: no hay camino (otra componente)


## 3. `BFS_complete`: encontrar todas las componentes

Una sola llamada a `BFS` recorre **una** componente. Para recorrer el grafo completo, Goodrich recorre todos los vértices y lanza un `BFS` nuevo desde cada vértice que **todavía no** ha sido descubierto. Cada uno de esos vértices es la **raíz** de un árbol, y el resultado es un **bosque** (un árbol BFS por componente).

In [18]:
print(inspect.getsource(BFS_complete))

def BFS_complete(g):
  """Perform BFS for entire graph and return forest as a dictionary.

  Result maps each vertex v to the edge that was used to discover it.
  (vertices that are roots of a BFS tree are mapped to None).
  """
  forest = {}
  for u in g.vertices():
    if u not in forest:
      forest[u] = None            # u will be a root of a tree
      BFS(g, u, forest)
  return forest



Las raíces son los vértices que quedan asociados a `None`. **Hay una raíz por componente**, así que contarlas da el número de componentes conexas:

In [19]:
forest = BFS_complete(g)
raices = [v for v in forest if forest[v] is None]
print('raíces:', raices)
print('número de componentes conexas:', len(raices))
print('¿el grafo es conexo?', len(raices) == 1)

raíces: [A, G, K, M]
número de componentes conexas: 4
¿el grafo es conexo? False


Para saber **qué vértices** forman cada componente, basta lanzar un `BFS` desde cada raíz y quedarse con las claves de su `discovered`. Con eso también se responde en $O(1)$ la pregunta "¿están `u` y `v` conectados?": basta ver si tienen la misma raíz.

**Costo:** cada vértice se descubre una sola vez y cada arista se revisa dos veces (una desde cada extremo), así que encontrar todas las componentes cuesta $O(n + m)$.

In [9]:
def componentes(g):
    """Retorna una lista de componentes; cada una es la lista de sus vértices."""
    vistos = set()
    comps = []
    for u in g.vertices():
        if u not in vistos:                # u es la raíz de una componente nueva
            discovered = {u: None}
            BFS(g, u, discovered)
            comps.append(list(discovered))
            vistos.update(discovered)
    return comps


comps = componentes(g)
for i, c in enumerate(comps, 1):
    print(f'componente {i}: {c}')

# a qué componente pertenece cada vértice
comp_de = {v: i for i, c in enumerate(comps, 1) for v in c}

def conectados(u, v):
    return comp_de[u] == comp_de[v]

print()
print('¿A y F conectados?', conectados(V['A'], V['F']))
print('¿A y J conectados?', conectados(V['A'], V['J']))
print('¿K y M conectados?', conectados(V['K'], V['M']))

componente 1: [A, B, C, D, E, F]
componente 2: [G, H, I, J]
componente 3: [K, L]
componente 4: [M]

¿A y F conectados? True
¿A y J conectados? False
¿K y M conectados? False


## 4. DFS: el otro recorrido

El recorrido en profundidad (*depth-first search*) hace lo contrario: en vez de explorar por niveles, sigue un camino **lo más lejos posible** y solo retrocede (*backtracking*) cuando se queda sin vecinos nuevos. Goodrich lo escribe de forma **recursiva** (Código 14.5):

In [10]:
print(inspect.getsource(DFS))

def DFS(g, u, discovered):
  """Perform DFS of the undiscovered portion of Graph g starting at Vertex u.

  discovered is a dictionary mapping each vertex to the edge that was used to
  discover it during the DFS. (u should be "discovered" prior to the call.)
  Newly discovered vertices will be added to the dictionary as a result.
  """
  for e in g.incident_edges(u):    # for every outgoing edge from u
    v = e.opposite(u)
    if v not in discovered:        # v is an unvisited vertex
      discovered[v] = e            # e is the tree edge that discovered v
      DFS(g, v, discovered)        # recursively explore from v



`DFS` usa el **mismo** contrato que `BFS`: llena `discovered` con la arista que descubrió a cada vértice. Por eso también existe `DFS_complete`, y también sirve para contar componentes: **ambos** recorridos visitan exactamente la componente del vértice inicial. Lo que cambia es **la forma del árbol** y, por tanto, el camino que entrega `construct_path`:

In [26]:
print(inspect.getsource(Graph.Edge))

  class Edge:
    """Lightweight edge structure for a graph."""
    __slots__ = '_origin', '_destination', '_element'
  
    def __init__(self, u, v, x):
      """Do not call constructor directly. Use Graph's insert_edge(u,v,x)."""
      self._origin = u
      self._destination = v
      self._element = x
  
    def endpoints(self):
      """Return (u,v) tuple for vertices u and v."""
      return (self._origin, self._destination)
  
    def opposite(self, v):
      """Return the vertex that is opposite v on this edge."""
      if not isinstance(v, Graph.Vertex):
        raise TypeError('v must be a Vertex')
      return self._destination if v is self._origin else self._origin
      raise ValueError('v not incident to edge')
  
    def element(self):
      """Return element associated with this edge."""
      return self._element
  
    def __hash__(self):         # will allow edge to be a map/set key
      return hash( (self._origin, self._destination) )

    def __str__(self):
      

In [27]:
def DFS1(g:Graph, u:Graph.Vertex, discovered:dict):
    for i in g.incident_edges(u):
        a = i.opposite(u)
        if a not in discovered:
            discovered[a] = u
            DFS1(g, a, discovered)


In [29]:
discovered = {V['A']:None}
DFS1(g, V['A'], discovered)

In [30]:
discovered

{A: None, B: A, D: B, C: D, E: C, F: E}

In [32]:
d_dfs

{A: None,
 B: (A,B,None),
 D: (B,D,None),
 C: (C,D,None),
 E: (C,E,None),
 F: (E,F,None)}

In [31]:
d_bfs = {V['A']: None}
BFS(g, V['A'], d_bfs)
d_dfs = {V['A']: None}
DFS(g, V['A'], d_dfs)

print('BFS visita:', list(d_bfs))
print('DFS visita:', list(d_dfs))
print()
print('camino A -> F con BFS:', construct_path(V['A'], V['F'], d_bfs))
print('camino A -> F con DFS:', construct_path(V['A'], V['F'], d_dfs))

forest_dfs = DFS_complete(g)
print()
print('componentes según DFS_complete:', sum(1 for v in forest_dfs if forest_dfs[v] is None))

BFS visita: [A, B, C, D, E, F]
DFS visita: [A, B, D, C, E, F]

camino A -> F con BFS: [A, B, D, F]
camino A -> F con DFS: [A, B, D, C, E, F]

componentes según DFS_complete: 4


| | BFS | DFS |
|---|---|---|
| Estrategia | por niveles (onda) | lo más profundo posible, luego retrocede |
| Estructura | cola / lista de nivel (iterativo) | pila / recursión |
| Visita la componente de `s` | sí | sí |
| `construct_path` da el camino más corto (en aristas) | **sí** | no necesariamente |
| Costo | $O(n + m)$ | $O(n + m)$ |
| Usos típicos | distancias, camino más corto sin pesos, niveles | detectar ciclos, orden topológico, retroceso (*backtracking*) |

## 5. Aplicación: salir de un laberinto

Un laberinto es un grafo disfrazado:

- **vértice** = cada casilla libre, identificada por su `(fila, columna)`;
- **arista** = dos casillas libres que son vecinas (arriba, abajo, izquierda o derecha);
- los **muros** simplemente no son vértices.

![Un laberinto y su grafo](../assets/ch14_laberinto_grafo.png)

Con ese modelo, las preguntas sobre el laberinto se vuelven preguntas sobre el grafo:

| Pregunta del laberinto | Pregunta del grafo | Herramienta |
|---|---|---|
| ¿Se puede llegar de E a S? | ¿están E y S en la misma componente? | `BFS` / `DFS` |
| ¿Cuál es la ruta más corta? | camino con menos aristas de E a S | `BFS` + `construct_path` |
| ¿Hay zonas a las que no se puede entrar? | ¿hay más de una componente? | `BFS_complete` |

Observa el **cuarto cerrado** del centro: 4 casillas libres rodeadas de muros. En el grafo son una **componente aparte**, y ningún recorrido que parta de `E` llegará jamás allí.

### 5.1 Del laberinto al grafo

El laberinto se escribe como una lista de cadenas: `#` es muro, `.` casilla libre, `E` entrada y `S` salida. Para crear cada arista **una sola vez**, cada casilla se conecta solo con su vecina de la **derecha** y la de **abajo** (la de la izquierda y la de arriba ya la conectaron a ella).

In [7]:
LABERINTO = [
    '###############',
    '#E......#.....#',
    '#.####.##.###.#',
    '#.#........#..#',
    '#.#.####.#.#.##',
    '#...#..#.#...##',
    '###.#..#.###..#',
    '#...####.#....#',
    '#.#......#.##.#',
    '#.#.####...#S.#',
    '###############',
]


def laberinto_a_grafo(lab):
    """Retorna (g, V): un Graph de Goodrich y el dict (fila, col) -> Vertex."""
    g = Graph()
    V = {}
    for f, linea in enumerate(lab):
        for c, ch in enumerate(linea):
            if ch != '#':                                  # cada casilla libre es un vértice
                V[(f, c)] = g.insert_vertex((f, c))
    for (f, c), v in V.items():
        for vecino in ((f, c + 1), (f + 1, c)):             # derecha y abajo
            if vecino in V:
                g.insert_edge(v, V[vecino])
    return g, V


def buscar(lab, ch):
    """(fila, col) de la primera aparición del carácter ch."""
    for f, linea in enumerate(lab):
        c = linea.find(ch)
        if c != -1:
            return (f, c)


lab, VL = laberinto_a_grafo(LABERINTO)
entrada, salida = VL[buscar(LABERINTO, 'E')], VL[buscar(LABERINTO, 'S')]

print(lab.vertex_count(), 'casillas libres (vértices)')
print(lab.edge_count(), 'pasos posibles entre casillas (aristas)')
print('entrada:', entrada, '  salida:', salida)
print('vecinos de la casilla (3, 8):', [e.opposite(VL[(3, 8)]) for e in lab.incident_edges(VL[(3, 8)])])

71 casillas libres (vértices)
75 pasos posibles entre casillas (aristas)
entrada: (1, 1)   salida: (9, 12)
vecinos de la casilla (3, 8): [(3, 7), (3, 9), (4, 8)]


### 5.2 Resolverlo con BFS

La idea es exactamente la de la sección 2: soltar una "onda" desde la entrada. Todas las casillas a 1 paso se marcan primero, luego las que están a 2 pasos, etc. La primera vez que la onda toca la salida, lo hace por el camino **más corto**, y `construct_path` lo reconstruye caminando hacia atrás por las aristas de descubrimiento.

Para verlo, una función que imprime el laberinto marcando con `o` las casillas de un camino:

In [8]:
def dibujar(lab, camino=(), marca='o'):
    """Imprime el laberinto marcando las casillas del camino (lista de Vertex)."""
    celdas = {v.element() for v in camino}
    for f, linea in enumerate(lab):
        print(''.join(marca if (f, c) in celdas and ch == '.' else ch
                      for c, ch in enumerate(linea)))


discovered = {entrada: None}
BFS(lab, entrada, discovered)
camino_bfs = construct_path(entrada, salida, discovered)

print('¿hay salida?', salida in discovered)
print('pasos del camino más corto:', len(camino_bfs) - 1)
print()
dibujar(LABERINTO, camino_bfs)

¿hay salida? True
pasos del camino más corto: 21

###############
#Eooooo.#.....#
#.####o##.###.#
#.#...ooooo#..#
#.#.####.#o#.##
#...#..#.#ooo##
###.#..#.###oo#
#...####.#...o#
#.#......#.##o#
#.#.####...#So#
###############


Con `BFS_niveles` (sección 2) se ve la onda: el nivel de cada casilla es el **número mínimo de pasos** desde la entrada. La figura muestra esas distancias; la salida está en el nivel 21. Las casillas marcadas con ✗ son el cuarto cerrado: la onda nunca las toca.

![BFS y DFS en el laberinto](../assets/ch14_laberinto_bfs_dfs.png)

In [14]:
_, niveles_lab = BFS_niveles(lab, entrada)
for k, nivel in enumerate(niveles_lab[:6]):
    print(f'a {k} pasos: {nivel}')
print('...')
print(f'la onda dura {len(niveles_lab)} niveles; la salida está en el nivel',
      next(k for k, nivel in enumerate(niveles_lab) if salida in nivel))

a 0 pasos: [(1, 1)]
a 1 pasos: [(1, 2), (2, 1)]
a 2 pasos: [(1, 3), (3, 1)]
a 3 pasos: [(1, 4), (4, 1)]
a 4 pasos: [(1, 5), (5, 1)]
a 5 pasos: [(1, 6), (5, 2)]
...
la onda dura 22 niveles; la salida está en el nivel 21


### 5.3 ¿Y con DFS?

DFS **también** sale del laberinto: es lo que hace una persona que avanza por un pasillo hasta toparse con un callejón sin salida, y entonces retrocede hasta la última bifurcación para probar otro pasillo. Pero el camino que encuentra depende del orden en que prueba los vecinos, y en general **no** es el más corto:

In [15]:
discovered_dfs = {entrada: None}
DFS(lab, entrada, discovered_dfs)
camino_dfs = construct_path(entrada, salida, discovered_dfs)

print('pasos con DFS:', len(camino_dfs) - 1, '   pasos con BFS:', len(camino_bfs) - 1)
print()
dibujar(LABERINTO, camino_dfs, marca='x')

pasos con DFS: 43    pasos con BFS: 21

###############
#Exxxxx.#xxxxx#
#.####x##x###x#
#.#xxxx.xx.#xx#
#.#x####x#.#x##
#..x#..#x#..x##
###x#..#x###xx#
#..x####x#...x#
#.#xxxxxx#.##x#
#.#.####...#Sx#
###############


### 5.4 Zonas inaccesibles = componentes conexas

`BFS_complete` sobre el grafo del laberinto cuenta sus componentes: una es la zona a la que se llega desde la entrada; las demás son zonas encerradas por muros.

In [16]:
forest = BFS_complete(lab)
raices = [v for v in forest if forest[v] is None]
print('componentes del laberinto:', len(raices))

for c in componentes(lab):
    if entrada not in c:
        print('zona inaccesible desde E:', c)

componentes del laberinto: 2
zona inaccesible desde E: [(5, 5), (5, 6), (6, 5), (6, 6)]


### 5.5 Un laberinto sin salida

Si un muro bloquea el único paso hacia la salida, `S` queda en **otra componente**: `BFS` termina sin descubrirla y `construct_path` retorna la lista vacía. Tapamos la casilla `(7, 13)`, que es el cuello de botella de la esquina inferior derecha:

In [17]:
BLOQUEADO = list(LABERINTO)
BLOQUEADO[7] = BLOQUEADO[7][:13] + '#' + BLOQUEADO[7][14:]   # muro nuevo en (7, 13)
dibujar(BLOQUEADO)

lab2, VL2 = laberinto_a_grafo(BLOQUEADO)
e2, s2 = VL2[buscar(BLOQUEADO, 'E')], VL2[buscar(BLOQUEADO, 'S')]
d2 = {e2: None}
BFS(lab2, e2, d2)
print()
print('camino:', construct_path(e2, s2, d2))
print('¿hay salida?', s2 in d2)
print('componentes del laberinto bloqueado:', len(componentes(lab2)))

###############
#E......#.....#
#.####.##.###.#
#.#........#..#
#.#.####.#.#.##
#...#..#.#...##
###.#..#.###..#
#...####.#...##
#.#......#.##.#
#.#.####...#S.#
###############

camino: []
¿hay salida? False
componentes del laberinto bloqueado: 3


## 6. Resumen y ejercicios

- Una **componente conexa** es un conjunto maximal de vértices conectados entre sí. Un grafo es conexo si y solo si tiene **una** componente.
- `BFS(g, s, discovered)` explora **por niveles**; el nivel de un vértice es su distancia (en aristas) a `s`. Solo visita la componente de `s`.
- `discovered` guarda, para cada vértice, la arista que lo descubrió; con `construct_path` se reconstruye el camino, que en BFS es **el más corto** en número de aristas.
- `BFS_complete(g)` (o `DFS_complete(g)`) lanza un recorrido desde cada vértice no descubierto: el número de **raíces** es el número de **componentes**. Costo $O(n + m)$.
- Un **laberinto** es un grafo (casillas libres + vecindad). BFS desde la entrada da la ruta más corta, DFS da *una* ruta, y las zonas encerradas son otras componentes.

**Patrón para usar los recorridos de Goodrich:**

```python
discovered = {s: None}                 # la raíz no tiene arista de descubrimiento
BFS(g, s, discovered)                  # o DFS(g, s, discovered)
v in discovered                        # ¿s alcanza a v?
construct_path(s, v, discovered)       # el camino (el más corto si fue BFS)
```

**Ejercicios**

1. Modifica `BFS_niveles` para que retorne un diccionario `vértice -> distancia` en vez de la lista de niveles.
2. Agrega movimientos en **diagonal** en `laberinto_a_grafo` (cada casilla tiene hasta 8 vecinos). ¿Cuántos pasos tiene ahora el camino más corto?
3. En un laberinto con varias salidas `S`, ¿cómo encontrarías la salida **más cercana** con un solo BFS?
4. Escribe una función que diga cuántas casillas tiene la componente **más grande** de un laberinto.
5. ¿Por qué `DFS` recursivo puede fallar con `RecursionError` en un laberinto de $1000 \times 1000$ y `BFS` no?